# Notebook 15 — Self-attention "from scratch" con NumPy

**Modulo:** 15 — Come funzionano davvero gli LLM
**Tempo stimato:** 1,5-2 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Una frase giocattolo trasformata in token, embedding e positional encoding, passo per passo
- Un'implementazione minimale di self-attention (Query, Key, Value) scritta a mano con NumPy
- Una visualizzazione numerica e grafica dell'attention map: cosa "guarda" ogni token
- Un'estensione a multi-head attention con due teste
- Un'implementazione della maschera causale (masked attention) usata nel decoder

## Prerequisiti

```bash
pip install numpy matplotlib
```

> **Nota:** questo notebook **non usa nessuna API e non serve nessuna chiave**. Tutto il calcolo
> avviene in locale con NumPy: l'obiettivo e vedere con i tuoi occhi i numeri dietro il meccanismo
> di attenzione descritto nella dispensa, non addestrare un modello vero.

---
## Cella 0 — Setup

Fissiamo un seed casuale per rendere l'esperimento riproducibile: i pesi di Query/Key/Value
verranno inizializzati casualmente (in un Transformer vero sarebbero appresi durante
l'addestramento), ma vogliamo ottenere sempre gli stessi numeri per poterli commentare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
np.set_printoptions(precision=3, suppress=True)


---
## Cella 1 — La frase giocattolo: tokenizzazione manuale

Usiamo una frase breve in italiano e la spezziamo in token semplicemente dividendo per spazi
(un vero tokenizzatore userebbe sotto-parole, ma per l'intuizione basta questo). Costruiamo il
vocabolario a mano, come nella dispensa (Sezione 3.1).

In [ ]:
frase = "il gatto nero dorme sul tappeto"
token = frase.split()

vocabolario = {parola: indice for indice, parola in enumerate(token)}
n_token = len(token)

print("Token:", token)
print("Vocabolario (parola -> indice):", vocabolario)
print("Numero di token nella sequenza:", n_token)


---
## Cella 2 — Embedding: da token a vettori

In un Transformer vero, l'embedding di ogni parola si impara durante l'addestramento. Qui, per
isolare il meccanismo di attenzione, generiamo embedding **casuali** di dimensione `d_model = 8`:
non hanno nessun significato semantico reale, servono solo a dare a ogni token un vettore su cui
far lavorare l'attenzione.

In [ ]:
d_model = 8  # dimensione dell'embedding di ogni token

embedding = np.random.randn(n_token, d_model)

print("Forma della matrice di embedding (n_token, d_model):", embedding.shape)
print("\nEmbedding del token 'gatto':")
print(embedding[token.index("gatto")])


---
## Cella 3 — Positional encoding

Come spiegato nella dispensa (Sezione 3.2), il Transformer elabora tutti i token in parallelo e ha
quindi bisogno di un segnale esplicito sulla posizione di ciascun token. Implementiamo la formula
originale basata su seno e coseno:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Poi sommiamo il positional encoding all'embedding, come fa l'architettura originale.

In [ ]:
def positional_encoding(lunghezza_sequenza: int, d_model: int) -> np.ndarray:
    posizioni = np.arange(lunghezza_sequenza)[:, np.newaxis]      # forma (seq_len, 1)
    dimensioni = np.arange(d_model)[np.newaxis, :]                 # forma (1, d_model)

    angoli = posizioni / np.power(10000, (2 * (dimensioni // 2)) / d_model)

    pe = np.zeros((lunghezza_sequenza, d_model))
    pe[:, 0::2] = np.sin(angoli[:, 0::2])  # indici pari -> seno
    pe[:, 1::2] = np.cos(angoli[:, 1::2])  # indici dispari -> coseno
    return pe


pe = positional_encoding(n_token, d_model)
X = embedding + pe  # rappresentazione finale: embedding semantico + informazione di posizione

print("Positional encoding (prime 3 posizioni):")
print(pe[:3])
print("\nForma di X (embedding + posizione):", X.shape)


---
## Cella 4 — Le matrici Query, Key, Value

Come descritto nella dispensa (Sezione 4.1), Q, K e V si ottengono applicando tre trasformazioni
lineari (matrici di pesi) alla rappresentazione `X`. In un modello vero questi pesi si imparano
durante l'addestramento; qui li inizializziamo casualmente, ma con dimensioni realistiche.

In [ ]:
d_k = 4  # dimensione dei vettori Q e K (spesso d_k = d_model / numero di teste)
d_v = 4  # dimensione dei vettori V

W_q = np.random.randn(d_model, d_k) * 0.1
W_k = np.random.randn(d_model, d_k) * 0.1
W_v = np.random.randn(d_model, d_v) * 0.1

Q = X @ W_q
K = X @ W_k
V = X @ W_v

print("Forma di Q:", Q.shape)
print("Forma di K:", K.shape)
print("Forma di V:", V.shape)


---
## Cella 5 — Calcolo dell'attention map

Seguiamo esattamente i passi descritti nella dispensa (Sezione 4.2):

```
Attention(Q, K, V) = softmax( Q · Kᵀ / √d_k ) · V
```

Prima calcoliamo gli attention score grezzi (`Q @ K.T`), poi li scaliamo dividendo per `√d_k`,
poi applichiamo softmax riga per riga per ottenere una vera distribuzione di probabilita.

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x_stabile = x - np.max(x, axis=axis, keepdims=True)  # per stabilita numerica
    esponenziali = np.exp(x_stabile)
    return esponenziali / np.sum(esponenziali, axis=axis, keepdims=True)


def self_attention(Q, K, V, maschera=None):
    """Calcola l'attenzione: score grezzi -> scalatura -> (maschera opzionale) -> softmax -> pesatura di V.
    Ritorna sia l'output sia la matrice di attention (utile per ispezionarla/visualizzarla)."""
    d_k = Q.shape[-1]
    score_grezzi = Q @ K.T
    score_scalati = score_grezzi / np.sqrt(d_k)

    if maschera is not None:
        score_scalati = score_scalati + maschera  # la maschera aggiunge -inf dove serve nascondere

    attention_map = softmax(score_scalati, axis=-1)
    output = attention_map @ V
    return output, attention_map


output, attention_map = self_attention(Q, K, V)

print("Attention map (righe = token che 'guarda', colonne = token 'guardato'):")
print(attention_map)
print("\nOgni riga somma a 1 (e' una distribuzione di probabilita):", attention_map.sum(axis=1))


---
## Cella 6 — Visualizzare l'attention map

Trasformiamo la matrice di numeri in un'immagine, con i token sugli assi, per vedere in un colpo
d'occhio cosa "guarda" ogni parola della frase. Le celle piu chiare/scure indicano attention score
piu alti (dipende dalla colormap).

In [ ]:
def disegna_attention_map(attention_map: np.ndarray, token: list[str], titolo: str = "Attention map") -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    immagine = ax.imshow(attention_map, cmap="viridis")

    ax.set_xticks(range(len(token)))
    ax.set_yticks(range(len(token)))
    ax.set_xticklabels(token, rotation=45, ha="right")
    ax.set_yticklabels(token)
    ax.set_xlabel("Token guardato (Key)")
    ax.set_ylabel("Token che guarda (Query)")
    ax.set_title(titolo)

    # Annotiamo ogni cella con il valore numerico
    for i in range(len(token)):
        for j in range(len(token)):
            ax.text(j, i, f"{attention_map[i, j]:.2f}", ha="center", va="center",
                     color="white" if attention_map[i, j] < 0.5 else "black", fontsize=8)

    fig.colorbar(immagine, ax=ax, label="Attention score")
    fig.tight_layout()
    plt.show()


disegna_attention_map(attention_map, token, titolo="Self-attention (pesi casuali, non addestrati)")


**Prima di continuare, osserva la matrice:** con pesi Q/K/V inizializzati casualmente (non
addestrati), non aspettarti che l'attenzione rifletta relazioni linguistiche reali (per esempio
"gatto" che guarda "nero" perche e un aggettivo che lo descrive). In un modello vero, questi pesi si
aggiustano durante l'addestramento perche l'attention map rifletta relazioni semanticamente
sensate. Qui l'obiettivo e vedere **il meccanismo**, non un risultato linguisticamente corretto.

**Le tue osservazioni** (modifica questa cella):

- Quale token riceve, in media, piu attenzione dagli altri? ...
- La riga di un token somma sempre a 1: perche? (ripensa alla softmax) ...

---
## Cella 7 — Multi-head attention: due teste in parallelo

Come descritto nella dispensa (Sezione 4.3), l'attenzione "multi-testa" ripete lo stesso calcolo
piu volte in parallelo, con pesi diversi per ogni testa, poi concatena i risultati e li proietta
con un'ultima trasformazione lineare.

In [ ]:
def multi_head_attention(X: np.ndarray, n_teste: int, d_model: int, d_k: int, d_v: int):
    """Applica n_teste attenzioni indipendenti in parallelo e ne concatena l'output."""
    output_teste = []
    attention_maps = []

    for testa in range(n_teste):
        W_q = np.random.randn(d_model, d_k) * 0.1
        W_k = np.random.randn(d_model, d_k) * 0.1
        W_v = np.random.randn(d_model, d_v) * 0.1

        Q_testa = X @ W_q
        K_testa = X @ W_k
        V_testa = X @ W_v

        output_testa, attention_testa = self_attention(Q_testa, K_testa, V_testa)
        output_teste.append(output_testa)
        attention_maps.append(attention_testa)

    output_concatenato = np.concatenate(output_teste, axis=-1)  # forma (n_token, n_teste * d_v)

    # Proiezione lineare finale, per riportare l'output alla dimensione d_model
    W_o = np.random.randn(output_concatenato.shape[-1], d_model) * 0.1
    output_finale = output_concatenato @ W_o

    return output_finale, attention_maps


output_multihead, mappe_per_testa = multi_head_attention(X, n_teste=2, d_model=d_model, d_k=4, d_v=4)

print("Forma dell'output multi-head (dopo la proiezione finale):", output_multihead.shape)

for i, mappa in enumerate(mappe_per_testa):
    disegna_attention_map(mappa, token, titolo=f"Testa {i + 1}")


**Le tue osservazioni** (modifica questa cella):

- Le due teste producono attention map identiche o diverse? Perche te lo aspettavi (ripensa a come
  sono inizializzati i pesi di ogni testa)? ...
- Nella dispensa si dice che teste diverse possono imparare a catturare relazioni diverse
  (grammaticali, semantiche, sintattiche). Con pesi casuali non addestrati, questo comportamento e
  visibile? Perche si o perche no? ...

---
## Esercizio — `# TODO`: implementare la maschera causale (masked attention)

Nella dispensa (Sezione 6.1) abbiamo visto perche il decoder ha bisogno di una maschera che
impedisca a ogni token di "vedere" i token futuri. La maschera si costruisce come una matrice
triangolare: 0 dove il token puo guardare (posizioni precedenti o se stesso), `-inf` dove non puo
(posizioni future) — cosi che, dopo la softmax, quelle posizioni ricevano probabilita zero.

Completa la funzione `crea_maschera_causale` qui sotto, poi passa il risultato a `self_attention`
tramite il parametro `maschera` che abbiamo gia predisposto nella Cella 5.

In [ ]:
def crea_maschera_causale(n_token: int) -> np.ndarray:
    """Ritorna una matrice (n_token, n_token) con 0 sotto/sulla diagonale e -inf sopra la diagonale."""
    # TODO: costruisci la maschera. Suggerimento: np.triu(np.ones((n_token, n_token)), k=1)
    # produce una matrice con 1 sopra la diagonale (esclusa) e 0 altrove: moltiplicala per
    # un valore molto negativo (es. -1e9) per ottenere l'effetto "-inf" in modo numericamente stabile.
    maschera = np.zeros((n_token, n_token))  # sostituisci con la maschera triangolare corretta
    return maschera


maschera_causale = crea_maschera_causale(n_token)

# Prova la tua implementazione (decommenta dopo aver completato il TODO)
# output_mascherato, attention_map_mascherata = self_attention(Q, K, V, maschera=maschera_causale)
# disegna_attention_map(attention_map_mascherata, token, titolo="Self-attention mascherata (stile decoder)")


**Cosa aspettarti se implementi correttamente la maschera:** confrontando questa attention map con
quella della Cella 6, dovresti vedere che la parte **triangolare superiore** (un token che guarda
un token che viene dopo di lui nella frase) e diventata zero. Ogni token puo ancora guardare se
stesso e tutti i token che lo precedono, ma non quelli che lo seguono — esattamente il comportamento
autoregressivo descritto nella dispensa.

---
## Sfida finale — Confrontare l'attenzione su due frasi diverse

Metti insieme tutti i pezzi costruiti in questo notebook in un'unica funzione riutilizzabile, e
usala per confrontare l'attention map di due frasi diverse.

1. Scrivi una funzione `analizza_attenzione(frase: str, d_model: int = 8, d_k: int = 4, d_v: int = 4)`
   che, data una frase, esegua l'intera pipeline (tokenizzazione, embedding casuale, positional
   encoding, calcolo di Q/K/V, self-attention) e ritorni `(token, attention_map)`.
2. Applicala a due frasi diverse, per esempio `"il gatto nero dorme sul tappeto"` e
   `"il cane grande corre nel parco"`, e disegna le due attention map una accanto all'altra.
3. Rifletti per iscritto: le due matrici, pur derivando da pesi casuali indipendenti, hanno una
   struttura simile (per esempio, quanto un token "guarda" se stesso)? Cosa cambierebbe con pesi
   *addestrati* invece che casuali?

In [ ]:
def analizza_attenzione(frase: str, d_model: int = 8, d_k: int = 4, d_v: int = 4):
    # TODO: implementa l'intera pipeline riusando le funzioni definite sopra in questo notebook
    # (positional_encoding, self_attention) e i passi delle Celle 1-5.
    token = frase.split()
    n_token = len(token)

    # TODO: genera embedding casuale (n_token, d_model)
    # TODO: somma il positional encoding
    # TODO: genera W_q, W_k, W_v casuali e calcola Q, K, V
    # TODO: chiama self_attention(Q, K, V) e ritorna (token, attention_map)

    return token, None  # sostituisci None con l'attention_map calcolata


# Prova la tua implementazione (decommenta dopo aver completato i TODO)
# token_1, mappa_1 = analizza_attenzione("il gatto nero dorme sul tappeto")
# token_2, mappa_2 = analizza_attenzione("il cane grande corre nel parco")
# disegna_attention_map(mappa_1, token_1, titolo="Frase 1")
# disegna_attention_map(mappa_2, token_2, titolo="Frase 2")


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 15 — Modulo 15 del Corso Completo di IA Generativa*